In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
import catboost as cb

In [2]:
dtm_today = datetime.today()
print(f'Latest run date: {dtm_today}')

Latest run date: 2025-02-28 18:27:25.553061


#### Functions

In [3]:
def get_catboost_feat_importance(cls_model_inference, pool_eval):
    df_tmp = cls_model_inference.get_feature_importance(
        data=pool_eval,
        type='LossFunctionChange',
        prettified=True,
    )
    df_tmp.columns = ['feature', 'importance']
    df_tmp.sort_values(by='importance', ascending=False, inplace=True)
    return df_tmp

#### Constants

In [4]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

str_target = 'Early_Pay_Delinquency_60_720_Flag'

Project: 20241112-simple-model-test
Task: ad_hoc


#### Output dir

In [5]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [6]:
list_str_df = [
    'train',
    'test',
    'holdout',
]
list_df = []
for str_df in tqdm(list_str_df):
    str_filename = f'df_{str_df}.gzip'
    str_uri = f's3://{str_project}/09_60_in_720/01_data_split/df_{str_df}.gzip'
    df = pd.read_parquet(str_uri)
    list_df.append(df)
df = pd.concat(list_df)
del list_df
# show
df

100%|██████████| 3/3 [00:02<00:00,  1.15it/s]


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg,request_month,row,prop_row,data_set,train,test,holdout
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0.616667,2021-07-01,1,0.000024,train,1,0,0
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,0,NaN,2021-07-01,2,0.000048,train,1,0,0
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,NaN,2021-07-01,3,0.000072,train,1,0,0
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0,1,0.000000,2021-07-01,4,0.000096,train,1,0,0
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0,1,0.000000,2021-07-01,5,0.000120,train,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41693,6453589,2022-11-30 23:45:32+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-11-30.gzip,1,1,Michigan,Franchise,Michigan,False,...,0,1,0.000000,2022-11-01,41694,0.999904,holdout,0,0,1
41694,6418453,2022-11-30 23:48:36+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-11-30.gzip,1,1,Michigan,Independent,Michigan,False,...,0,0,0.000000,2022-11-01,41695,0.999928,holdout,0,0,1
41695,6411914,2022-11-30 23:53:12+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-11-30.gzip,1,1,Washington,Independent,Washington,True,...,0,0,NaN,2022-11-01,41696,0.999952,holdout,0,0,1
41696,6434554,2022-11-30 23:57:04+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-11-30.gzip,1,1,Michigan,Franchise,Michigan,False,...,0,0,0.137374,2022-11-01,41697,0.999976,holdout,0,0,1


#### Impute ENG-wtd_avg

In [7]:
df['ENG-wtd_avg'] = df['ENG-wtd_avg'].fillna(0)
# subset
df = df[df['ENG-wtd_avg'] < 0.5].copy()
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg,request_month,row,prop_row,data_set,train,test,holdout
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,0,NaN,2021-07-01,2,0.000048,train,1,0,0
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,NaN,2021-07-01,3,0.000072,train,1,0,0
8,5709029,2021-07-27 09:42:40.7808217,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Missouri,Franchise,Missouri,False,...,0,0,NaN,2021-07-01,9,0.000216,train,1,0,0
9,5709609,2021-07-27 09:47:57.8551698,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Indiana,Franchise,Indiana,True,...,0,0,0.000000,2021-07-01,11,0.000264,train,1,0,0
12,5710164,2021-07-27 10:14:00.4561057,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Ohio,Independent,Ohio,False,...,0,0,NaN,2021-07-01,12,0.000288,train,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41692,6450520,2022-11-30 23:33:56+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-11-30.gzip,1,1,Illinois,Franchise,Wisconsin,True,...,0,0,0.416667,2022-11-01,41693,0.999880,holdout,0,0,1
41693,6453589,2022-11-30 23:45:32+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-11-30.gzip,1,1,Michigan,Franchise,Michigan,False,...,0,1,0.000000,2022-11-01,41694,0.999904,holdout,0,0,1
41694,6418453,2022-11-30 23:48:36+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-11-30.gzip,1,1,Michigan,Independent,Michigan,False,...,0,0,0.000000,2022-11-01,41695,0.999928,holdout,0,0,1
41695,6411914,2022-11-30 23:53:12+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-11-30.gzip,1,1,Washington,Independent,Washington,True,...,0,0,NaN,2022-11-01,41696,0.999952,holdout,0,0,1


#### Get features

In [8]:
# rm target
list_cols_model = [col for col in df.columns if col != str_target]
# rm non-numeric
list_cols_object = [col for col in list_cols_model if df[col].dtype not in ['int64','float64']]
list_cols_model = [col for col in df.columns if col not in list_cols_object]
# rm tags
list_cols_model = [col for col in list_cols_model if 'tag' not in col.lower()]
# rm flag
list_cols_model = [col for col in list_cols_model if 'flag' not in col.lower()]
# rm loss
list_cols_model = [col for col in list_cols_model if 'loss' not in col.lower()]
# rm cols to ignore
list_cols_ignore = [
    'data_set',
    'sum',
    'accountid',
    'bigdebtorid__app',
    'days_on_books',
    'bigdebtorid__ln',
    'bigaccountid__ln',
    'strzipcode__app',
    'bitdefault__app',
    'dti__app',
    'pti__app',
    'prop_row',
    'fltapproveddebttoincome__app',
    'fltapprovedapr_contract__app',
    'fltacquisitionfee__app',
    'row',
    'payment__app',
]
list_cols_model = [col for col in list_cols_model if col not in list_cols_ignore]

#### Fit model

In [9]:
# get train and test
df_train = df[df['data_set'] == 'train'].copy()
df_test = df[df['data_set'] == 'test'].copy()

# pool data
pool_train = cb.Pool(
    df_train[list_cols_model].copy(), 
    df_train[str_target], 
)
# pool
pool_valid = cb.Pool(
    df_test[list_cols_model].copy(), 
    df_test[str_target], 
)
# init class
cls_model_inference = cb.CatBoostClassifier(
    task_type='CPU',
    nan_mode='Min',
    random_state=42,
    eval_metric='AUC',
    iterations=100,
    learning_rate=None,
    class_weights=None,
    depth=None,
)
# fit
cls_model_inference.fit(
    pool_train,
    eval_set=[pool_valid],
    verbose=True,
    use_best_model=True,
    early_stopping_rounds=10, 
)

Learning rate set to 0.1637
0:	test: 0.5749964	best: 0.5749964 (0)	total: 84ms	remaining: 8.32s
1:	test: 0.5855460	best: 0.5855460 (1)	total: 111ms	remaining: 5.43s
2:	test: 0.5967104	best: 0.5967104 (2)	total: 136ms	remaining: 4.4s
3:	test: 0.6012653	best: 0.6012653 (3)	total: 161ms	remaining: 3.87s
4:	test: 0.6017204	best: 0.6017204 (4)	total: 186ms	remaining: 3.54s
5:	test: 0.6048700	best: 0.6048700 (5)	total: 212ms	remaining: 3.33s
6:	test: 0.6147031	best: 0.6147031 (6)	total: 239ms	remaining: 3.17s
7:	test: 0.6167581	best: 0.6167581 (7)	total: 264ms	remaining: 3.03s
8:	test: 0.6185413	best: 0.6185413 (8)	total: 290ms	remaining: 2.94s
9:	test: 0.6188220	best: 0.6188220 (9)	total: 316ms	remaining: 2.84s
10:	test: 0.6221149	best: 0.6221149 (10)	total: 342ms	remaining: 2.76s
11:	test: 0.6250063	best: 0.6250063 (11)	total: 368ms	remaining: 2.7s
12:	test: 0.6260483	best: 0.6260483 (12)	total: 393ms	remaining: 2.63s
13:	test: 0.6269323	best: 0.6269323 (13)	total: 420ms	remaining: 2.58s
1

#### Get feature importance

In [10]:
# get importance
df_tmp = get_catboost_feat_importance(
    cls_model_inference=cls_model_inference,
    pool_eval=pool_valid,
)

#### Map description

In [11]:
# get the data dict
df_data_dict = pd.read_csv('data_dictionary.csv')
dict_map = dict(zip(df_data_dict['feature_name'], df_data_dict['Description']))
df_tmp['description'] = df_tmp['feature'].map(dict_map)

# save
str_filename = 'df_feat_imp.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp

,feature,importance,description
0,linka006__tu,0.001259,Number of total inquiries in the last 3 years....
1,fltadvance__app,0.000886,NaN
2,bi09s__tu,0.000811,Number of bank installment trades opened in pa...
3,balmag01__tu,0.000663,Non-mortgage balance magnitude algorithm over ...
4,au20s__tu,0.000636,Months since oldest auto trade opened
...,...,...,...
2642,g228s__tu,-0.000185,Number of 90 or more days past due trades (cur...
2643,agg613__tu,-0.000227,Aggregate bankcard credit line for month 13
2644,addrprevioustimeoldest__ln,-0.000241,Time (in months) since the subject was first r...
2645,cv05__tu,-0.000348,Months since most recent non-medical third par...
